# Phase 3: Visual/Spiral CNN Classifier
This notebook trains a Convolutional Neural Network (CNN) to classify spiral drawings as **Healthy** or **Parkinson's**.

**Dataset:**
- `data/spiral/training/` → 36 healthy + 36 parkinson images
- `data/spiral/testing/`  → 15 healthy + 15 parkinson images

**Image format:** RGB, 100×100 pixels

**Normalization approach:** Pixel values are normalized **outside** the model (via a `.map()` call), NOT via an internal `Rescaling` layer. This is critical to match how `backend/utils.py` preprocesses images at inference time — it already divides by 255.0 before calling `model.predict()`.

In [ ]:
import os
import tensorflow as tf
from tensorflow.keras import layers, models

# ==========================================
# STEP 1: DEFINE DATA DIRECTORIES
# ==========================================
# Paths relative to the project root (run this notebook from there, or adjust the path)
train_dir = '../data/spiral/training'
test_dir  = '../data/spiral/testing'

print('Training dir exists:', os.path.exists(train_dir))
print('Testing dir exists: ', os.path.exists(test_dir))

# ==========================================
# STEP 2: LOAD THE IMAGES
# label_mode='binary' → 0 or 1 (2-class problem)
# class_names determined alphabetically: 'healthy'=0, 'parkinson'=1
# ==========================================
train_dataset = tf.keras.utils.image_dataset_from_directory(
    train_dir, image_size=(100, 100), batch_size=16, label_mode='binary', shuffle=True, seed=42)
test_dataset  = tf.keras.utils.image_dataset_from_directory(
    test_dir,  image_size=(100, 100), batch_size=16, label_mode='binary', shuffle=False)

print('Class names:', train_dataset.class_names, '(healthy=0, parkinson=1)')
print('Training batches:', len(train_dataset))
print('Testing batches: ', len(test_dataset))

# ==========================================
# STEP 3: NORMALIZE PIXEL VALUES (outside the model)
# IMPORTANT: Do NOT use a Rescaling layer inside the model.
# backend/utils.py already divides by 255.0 before calling model.predict().
# If we put Rescaling inside the model, inference would double-normalize (bug!)
# ==========================================
normalization_layer = layers.Rescaling(1./255)
train_dataset = train_dataset.map(lambda x, y: (normalization_layer(x), y))
test_dataset  = test_dataset.map(lambda x, y: (normalization_layer(x), y))

# ==========================================
# STEP 4: BUILD THE CNN MODEL
# ==========================================
model = models.Sequential([
    layers.Input(shape=(100, 100, 3)),         # 100x100 RGB input

    # Layer 1: Detect basic edges/curves
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    # Layer 2: Detect complex tremor patterns
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    # Classifier head
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(1, activation='sigmoid')      # Output: 0=Healthy, 1=Parkinson's
])

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

model.summary()

# ==========================================
# STEP 5: TRAIN THE MODEL (15 epochs)
# ==========================================
print('\n--- Starting training (15 epochs) ---')
history = model.fit(train_dataset, epochs=15, validation_data=test_dataset, verbose=1)

# ==========================================
# STEP 6: FINAL EVALUATION
# ==========================================
loss, accuracy = model.evaluate(test_dataset, verbose=0)
print(f'\n--- MODULE 3 FINAL RESULTS ---')
print(f'Spiral CNN Accuracy: {accuracy * 100:.2f}%')
print(f'Loss: {loss:.4f}')

# ==========================================
# STEP 7: SAVE THE MODEL (run this only to update backend/models/)
# ==========================================
# Uncomment the line below to overwrite the production model:
# model.save('../backend/models/spiral_model.h5')
# print('Model saved to ../backend/models/spiral_model.h5')